# 07 — Optimizer Ablation Study
## Hybrid Quantum-Classical NLP Framework

**⚠️ Run Cell 1 → Restart Runtime → Run Cell 2 onwards**

In [ ]:
# ============================================================
# CELL 1 — Installation  (then Runtime → Restart Runtime)
# ============================================================
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.strip())
    return r

print("Step 1/6: Wiping stale packages from disk...")
run("pip uninstall -y pennylane pennylane-lightning autoray jax jaxlib 2>/dev/null || true")
run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'pennylane*' -exec rm -rf {} + 2>/dev/null || true")
run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'PennyLane*' -exec rm -rf {} + 2>/dev/null || true")
run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'autoray*'   -exec rm -rf {} + 2>/dev/null || true")
print("  ✓ Stale files wiped")

print("Step 2/6: Installing numpy + scipy (ABI-compatible pair)...")
run("pip install -q --no-cache-dir numpy==1.26.4")
run("pip install -q --no-cache-dir scipy==1.13.1")
print("  ✓ numpy/scipy OK")

print("Step 3/6: Pinning JAX to a pennylane-compatible version...")
run("pip install -q --no-cache-dir 'jax==0.4.26' 'jaxlib==0.4.26'")
print("  ✓ JAX pinned")

print("Step 4/6: Installing autoray + pennylane (matched pair)...")
run("pip install -q --no-cache-dir autoray==0.6.12")
run("pip install -q --no-cache-dir pennylane==0.38.0 pennylane-lightning==0.38.0")
print("  ✓ pennylane/autoray OK")

print("Step 5/6: Installing NLP/ML stack...")
run("pip install -q --no-cache-dir "
    "numba>=0.57.1 "
    "transformers>=4.40.0 "
    "datasets>=2.19.0 "
    "scikit-learn>=1.4.0 "
    "gensim>=4.3.0 "
    "nltk>=3.8.0 "
    "seaborn>=0.13.0 "
    "tabulate>=0.9.0 "
    "pyyaml>=6.0 "
    "pandas==2.2.2 "
    "requests==2.32.4 "
    "'rich>=12.4.4,<14' "
    "'cachetools>=4.2.1,<7' "
    "'pillow<12.0' "
    "'tomlkit>=0.12.0,<0.14.0'"
)
print("  ✓ NLP/ML stack OK")

print("Step 6/6: Verifying installed versions...")
for pkg in ["pennylane", "autoray", "numpy", "scipy", "jax"]:
    run(f"pip show {pkg} | grep -E 'Name|Version'")

print()
print("=" * 60)
print("✅ Done! NOW: Runtime → Restart Runtime")
print("   Then run Cell 2 onwards.")
print("=" * 60)


In [ ]:
# ============================================================
# CELL 2 — Verify Imports  (AFTER restarting runtime)
# ============================================================
import importlib

print("Verifying all imports...")
print("-" * 50)

ok = True
checks = [
    ("numpy",       "1.26"),
    ("scipy",       "1.13"),
    ("jax",         "0.4.26"),
    ("autoray",     "0.6"),
    ("pennylane",   "0.38"),
    ("torch",       None),
    ("transformers",None),
    ("sklearn",     None),
]

for mod, prefix in checks:
    try:
        m   = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        chk = prefix is None or ver.startswith(prefix)
        print(f"  {'✅' if chk else '⚠️ '} {mod:<15} {ver}")
        if not chk: ok = False
    except Exception as e:
        print(f"  ❌ {mod:<15} {e}")
        ok = False

print()
try:
    import pennylane as qml
    dev = qml.device("default.qubit", wires=2)
    @qml.qnode(dev)
    def _t():
        qml.Hadamard(wires=0)
        qml.CNOT(wires=[0,1])
        return qml.expval(qml.PauliZ(0))
    print(f"  ✅ pennylane circuit  result={float(_t()):.4f}")
except Exception as e:
    print(f"  ❌ pennylane circuit  {e}")
    ok = False

print()
if ok:
    print("✅ All checks passed — safe to continue!")
else:
    print("❌ Checks failed.")
    print("   → Runtime → Factory Reset Runtime, then rerun Cell 1.")


In [ ]:
# ============================================================
# CELL 3 — Mount Google Drive
# ============================================================
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/HybridQuantumNLP"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f"✅ Drive mounted. Project dir: {DRIVE_PROJECT_DIR}")


In [ ]:
# ============================================================
# CELL 4A — Load Project: Upload ZIP  (run this OR 4B)
# ============================================================
from google.colab import files
import zipfile, glob, os, sys

print("Upload your project zip...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
PROJECT_DIR = "/content/project"

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(PROJECT_DIR)

run_files = glob.glob(f"{PROJECT_DIR}/**/run.py", recursive=True)
if run_files:
    PROJECT_DIR = os.path.dirname(run_files[0])

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"✅ Project loaded: {PROJECT_DIR}")


In [ ]:
# ============================================================
# CELL 4B — Load Project: From Drive  (run this OR 4A)
# ============================================================
import os, sys, shutil, zipfile

DRIVE_ZIP_PATH = "/content/drive/MyDrive/HybridQuantumNLP/JU_project.zip"
PROJECT_DIR    = "/content/project"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as z:
    z.extractall(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"✅ Project loaded from Drive: {PROJECT_DIR}")


In [ ]:
# ============================================================
# CELL 5 — Configuration
# ============================================================
import torch

LANGUAGE         = "english"
# ⬇️ SET THESE to the results from notebooks 04, 05, and 06
BEST_EMBEDDING   = "tfidf"       # Update from 04_embedding_study
BEST_VQC         = "v2"          # Update from 05_vqc_ablation_study
BEST_ACTIVATION  = "tanh"        # Update from 06_activation_ablation
N_QUBITS         = 4
N_LAYERS         = 2
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
MAX_SAMPLES      = None
QUANTUM_EPOCHS   = 30
LEARNING_RATE    = 0.01

print(f"Language:         {LANGUAGE}")
print(f"Best Embedding:   {BEST_EMBEDDING}")
print(f"Best VQC:         {BEST_VQC}")
print(f"Best Activation:  {BEST_ACTIVATION}")
print(f"Qubits:           {N_QUBITS}")
print(f"Device:           {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Max samples:      {MAX_SAMPLES or 'Full dataset'}")


In [ ]:
# ============================================================
# CELL 6 — Build Config Object
# ============================================================
from config import load_config

cfg = load_config(overrides={
    "device": DEVICE,
    "embeddings": {"method": BEST_EMBEDDING},
    "quantum_template": {"n_qubits": N_QUBITS, "n_layers": N_LAYERS, "epochs": QUANTUM_EPOCHS},
    "custom_vqc":       {"n_qubits": N_QUBITS, "n_layers": N_LAYERS, "epochs": QUANTUM_EPOCHS,
                         "version": BEST_VQC, "learning_rate": LEARNING_RATE},
})
print("✅ Config ready.")


In [ ]:
# ============================================================
# CELL 7 — Dataset Overview
# ============================================================
from data.registry import get_dataset_info, UNIFIED_LABELS

print("=" * 65)
for name, lang in [("go_emotions","English"),("emo_in_hindi","Hindi"),("emonoba","Bengali")]:
    info = get_dataset_info(name)
    src  = info.hf_id or info.github_url or info.path
    print(f"  {lang}: {info.name}")
    print(f"    Source: {info.source} ({src})")
    print(f"    Labels: {info.num_labels} → {info.label_names[:6]}")
print(f"  Unified labels: {UNIFIED_LABELS}")
print("=" * 65)


In [ ]:
# ============================================================
# CELL 8 — Dataset Load Test
# ============================================================
from data.loader import load_dataset_for_language

test_ds = load_dataset_for_language(LANGUAGE, cfg, max_samples=100)
print(test_ds.summary())
print(f"Labels: {test_ds.label_names}")
print(f"Sample: {test_ds.train['texts'][0][:120]}...")
print("✅ Dataset OK")


---
## Optimizer Ablation Study

**Fixed:**
- Embedding = BEST_EMBEDDING (from notebook 04)
- VQC = BEST_VQC (from notebook 05)
- Activation = BEST_ACTIVATION (from notebook 06)

**Variable:**
- Adam
- AdamW
- SGD (with momentum=0.9)
- RMSProp

In [ ]:
# ============================================================
# CELL 10 — Optimizer Comparison Loop
# ============================================================
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

from data.loader import load_dataset_for_language
from preprocessing.pipeline import preprocess_dataset
from embeddings.factory import get_embedder, embed_dataset
from models.custom_vqc.hybrid_model import CustomVQCModel, CustomVQCClassifier
from models.custom_vqc.feature_compressor import FeatureCompressor
from evaluation.metrics import compute_metrics
from visualization.plots import print_results_table, plot_model_comparison

# --- Optimizer factory ---
def get_optimizer(name, params, lr):
    """Create optimizer by name."""
    name_lower = name.lower()
    if name_lower == "adam":
        return torch.optim.Adam(params, lr=lr)
    elif name_lower == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    elif name_lower == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    elif name_lower == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr)
    else:
        raise ValueError(f"Unknown optimizer: {name}")

# --- Custom training loop with optimizer override ---
def train_with_optimizer(model, optimizer_name, lr, X_train, y_train,
                          X_val, y_val, epochs, batch_size, device):
    """Train a CustomVQCModel with a specific optimizer.
    Reuses the same training logic as CustomVQCClassifier.train()
    but allows optimizer injection."""
    device = torch.device(device)
    model = model.to(device)

    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    )
    val_ds = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    optimizer = get_optimizer(optimizer_name, model.parameters(), lr)
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    start_time = time.perf_counter()

    for epoch in range(1, epochs + 1):
        # --- Training ---
        model.train()
        running_loss, running_acc, n_batches = 0.0, 0.0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False)
        for batch_x, batch_y in pbar:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            running_acc += (logits.argmax(dim=-1) == batch_y).float().mean().item()
            n_batches += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        history["train_loss"].append(running_loss / max(n_batches, 1))
        history["train_acc"].append(running_acc / max(n_batches, 1))

        # --- Validation ---
        model.eval()
        val_loss, val_acc, val_batches = 0.0, 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                val_loss += loss.item()
                val_acc += (logits.argmax(dim=-1) == batch_y).float().mean().item()
                val_batches += 1

        history["val_loss"].append(val_loss / max(val_batches, 1))
        history["val_acc"].append(val_acc / max(val_batches, 1))

        print(f"  Epoch {epoch}/{epochs} — "
              f"train_loss={history['train_loss'][-1]:.4f}  "
              f"val_loss={history['val_loss'][-1]:.4f}  "
              f"val_acc={history['val_acc'][-1]:.4f}")

    history["training_time"] = time.perf_counter() - start_time
    return history


def evaluate_model(model, X_test, y_test, batch_size, device):
    """Evaluate model, return metrics dict."""
    model.eval()
    device_t = torch.device(device)
    tensor_x = torch.tensor(X_test, dtype=torch.float32).to(device_t)
    all_preds = []
    with torch.no_grad():
        loader = DataLoader(TensorDataset(tensor_x), batch_size=batch_size, shuffle=False)
        for (batch_x,) in loader:
            logits = model(batch_x)
            all_preds.append(logits.argmax(dim=-1).cpu().numpy())
    preds = np.concatenate(all_preds)
    return compute_metrics(y_test, preds)


print("✅ Training utilities ready.")


In [ ]:
# ============================================================
# CELL 11 — Load Data & Run Optimizer Ablation
# ============================================================

OPTIMIZERS_TO_TEST = ["Adam", "AdamW", "SGD", "RMSProp"]

# Load data ONCE
print("Loading and embedding data...")
dataset = load_dataset_for_language(LANGUAGE, cfg, max_samples=MAX_SAMPLES)
dataset = preprocess_dataset(dataset, cfg)
embedder = get_embedder(cfg, LANGUAGE)
embeddings = embed_dataset(embedder, dataset)

X_train = embeddings["train"]
y_train = np.array(dataset.train["labels"])
X_val   = embeddings["val"]
y_val   = np.array(dataset.val["labels"])
X_test  = embeddings["test"]
y_test  = np.array(dataset.test["labels"])

input_dim  = X_train.shape[1]
num_labels = dataset.num_labels
batch_size = getattr(cfg.custom_vqc, "batch_size", 32)

print(f"  Data: train={len(y_train)}, val={len(y_val)}, test={len(y_test)}")
print(f"  Input dim: {input_dim}, Classes: {num_labels}")
print("✅ Data ready.")

# Map BEST_ACTIVATION label back to internal name
ACTIVATION_MAP = {"tanh": "tanh", "sigmoid": "sigmoid", "relu": "relu", "identity": "none"}
act_internal = ACTIVATION_MAP.get(BEST_ACTIVATION, BEST_ACTIVATION)

# If using relu, patch FeatureCompressor
if act_internal == "relu":
    FeatureCompressor._ACTIVATIONS["relu"] = nn.ReLU
    print("  Patched FeatureCompressor for relu")

# --- Run optimizer ablation ---
optimizer_results = {}

for opt_name in OPTIMIZERS_TO_TEST:
    print(f"\n{'='*60}")
    print(f"  Testing optimizer: {opt_name}")
    print(f"{'='*60}")

    try:
        # Build a fresh model for each optimizer
        model = CustomVQCModel(
            input_dim=input_dim,
            n_qubits=N_QUBITS,
            n_layers=N_LAYERS,
            num_classes=num_labels,
            version=BEST_VQC,
        )

        # Override activation if needed
        if act_internal != "tanh":
            model.compressor = FeatureCompressor(
                input_dim, N_QUBITS, activation=act_internal
            )

        model = model.to(DEVICE)

        # Train with specific optimizer
        history = train_with_optimizer(
            model, opt_name, LEARNING_RATE,
            X_train, y_train, X_val, y_val,
            QUANTUM_EPOCHS, batch_size, DEVICE
        )

        # Evaluate
        test_metrics = evaluate_model(model, X_test, y_test, batch_size, DEVICE)

        optimizer_results[opt_name] = {
            "history": history,
            "test_metrics": test_metrics,
            "params": sum(p.numel() for p in model.parameters()),
            "optimizer": opt_name,
        }
        print(f"  ✅ {opt_name}: F1={test_metrics['f1']:.4f}, Acc={test_metrics['accuracy']:.4f}")

    except Exception as e:
        print(f"  ❌ {opt_name} failed: {e}")
        import traceback; traceback.print_exc()
        optimizer_results[opt_name] = {"error": str(e)}

print("\n" + "="*60)
print("  Optimizer ablation complete!")
print("="*60)


In [ ]:
# ============================================================
# CELL 12 — Results Table & Comparison Plots
# ============================================================
import os
from visualization.plots import print_results_table, plot_model_comparison

os.makedirs("results/optimizer_ablation", exist_ok=True)

print_results_table(optimizer_results, title="Optimizer Comparison")

plot_model_comparison(
    optimizer_results, metric="f1",
    title=f"Optimizer Ablation — F1 ({LANGUAGE})",
    save_path="results/optimizer_ablation/optimizer_f1.png"
)
plot_model_comparison(
    optimizer_results, metric="accuracy",
    title=f"Optimizer Ablation — Accuracy ({LANGUAGE})",
    save_path="results/optimizer_ablation/optimizer_acc.png"
)


In [ ]:
# ============================================================
# CELL 13 — Convergence & Training Curves
# ============================================================
import matplotlib.pyplot as plt
from visualization.plots import plot_training_curves

# Individual training curves
for model_name, data in optimizer_results.items():
    if isinstance(data, dict) and "history" in data:
        h = data["history"]
        if "train_loss" in h and len(h["train_loss"]) > 1:
            plot_training_curves(
                h,
                title=f"Training Curves — {model_name}",
                save_path=f"results/optimizer_ablation/curves_{model_name}.png"
            )

# Overlay convergence plot (all optimizers on one chart)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for opt_name, data in optimizer_results.items():
    if isinstance(data, dict) and "history" in data:
        h = data["history"]
        if "train_loss" in h:
            axes[0].plot(h["train_loss"], label=opt_name, linewidth=2)
        if "val_loss" in h:
            axes[1].plot(h["val_loss"], label=opt_name, linewidth=2)

axes[0].set_title("Training Loss Convergence")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Validation Loss Convergence")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs("results/optimizer_ablation", exist_ok=True)
plt.savefig("results/optimizer_ablation/convergence_overlay.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ Convergence overlay saved.")


In [ ]:
# ============================================================
# CELL 14 — Determine Best Optimizer
# ============================================================
import json

best_f1 = -1
BEST_OPTIMIZER = "Adam"

for name, data in optimizer_results.items():
    if isinstance(data, dict) and "test_metrics" in data:
        f1 = data["test_metrics"].get("f1", 0)
        if f1 > best_f1:
            best_f1 = f1
            BEST_OPTIMIZER = name

print(f"\n{'='*60}")
print(f"  BEST OPTIMIZER: {BEST_OPTIMIZER} (F1={best_f1:.4f})")
print(f"{'='*60}")

results_summary = {
    "best_optimizer": BEST_OPTIMIZER,
    "best_f1": best_f1,
    "embedding_used": BEST_EMBEDDING,
    "vqc_used": BEST_VQC,
    "activation_used": BEST_ACTIVATION,
    "all_results": {
        k: v.get("test_metrics", {}) if isinstance(v, dict) else {}
        for k, v in optimizer_results.items()
    }
}

with open("results/optimizer_ablation/summary.json", "w") as f:
    json.dump(results_summary, f, indent=2, default=str)

print(f"\nResults saved to: results/optimizer_ablation/")
print(f"\n" + "="*60)
print(f"  FINAL BEST PIPELINE:")
print(f"    Embedding:  {BEST_EMBEDDING}")
print(f"    VQC:        {BEST_VQC}")
print(f"    Activation: {BEST_ACTIVATION}")
print(f"    Optimizer:  {BEST_OPTIMIZER}")
print(f"    F1 Score:   {best_f1:.4f}")
print(f"{'='*60}")


In [ ]:
# ============================================================
# CELL 15 — Save Results to Google Drive
# ============================================================
import os, shutil

results_src = os.path.join(PROJECT_DIR, "results", "optimizer_ablation")
results_dst = os.path.join(DRIVE_PROJECT_DIR, "results", "optimizer_ablation")

if os.path.exists(results_src):
    os.makedirs(results_dst, exist_ok=True)
    for root, dirs, files_list in os.walk(results_src):
        rel = os.path.relpath(root, results_src)
        dst = os.path.join(results_dst, rel)
        os.makedirs(dst, exist_ok=True)
        for f in files_list:
            shutil.copy2(os.path.join(root, f), os.path.join(dst, f))
    print(f"✅ Results saved to: {results_dst}")
else:
    print("⚠️  No results directory found — run experiments first.")
